Do the Magnificent 7 stocks have systematic risk (beta) that differs from the market portfolio as predicted by CAPM (Capital Asset Pricing Model)?

Beta (β) measures an investment's volatility or systematic risk compared to the overall market, typically the S&P 500, with the market's beta set at 1.0. A beta > 1 means the stock is more volatile (risky/rewarding), < 1 means less volatile, and = 1 means it moves with the market, helping investors assess how an asset adds risk to a diversified portfolio.

(Null Hypothesis)
For each Magnificent 7 stock i, the true CAPM beta equals 1:
$$H_0​: βi​=1$$
The stock has the same systematic risk as the market portfolio. Its returns move one-for-one with the market.

(Alternative Hypothesis)
For each Magnificent 7 stock i, the true CAPM beta is not equal to 1:
$$H_1​: βi​\ne 1$$
The stock’s systematic risk differs from the market, either more volatile $$(β>1)$$ or more defensive $$(β<1)$$

This study estimates the systematic risk (beta) of the Magnificent 7 stocks using the Capital Asset Pricing Model (CAPM). Excess returns for each stock are regressed on excess market returns using Ordinary Least Squares (OLS). The empirical specification is:

$$R_{i,t} - R_{f,t} = \alpha_i + \beta_i (R_{m,t} - R_{f,t}) + \epsilon_{i,t}$$

where R{i,t} is the return on stock i at time t, R{m,t} is the market return, and R{f,t} is the risk-free rate. Beta estimates measure each stock’s exposure to systematic market risk. Statistical inference focuses on testing whether estimated betas differ significantly from 1. Robustness checks include estimating betas using both daily and monthly returns, as well as alternative market proxies (S&P 500 and CRSP value-weighted market index), to assess the stability and reliability of beta estimates.

In [40]:
import yfinance as yf
import pandas as pd
import statsmodels.api as sm

MAG7 = ['AAPL', 'MSFT', 'AMZN', 'GOOGL', 'META', 'NVDA', 'TSLA']
MARKET_PROXY = 'SPY'          #S&P 500 ETF
START_DATE = '2015-01-01'
END_DATE = '2025-01-01'
RETURN_FREQ = 'monthly'       #'daily' or 'monthly'

#price data
tickers = MAG7 + [MARKET_PROXY]

prices = yf.download(
    tickers,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=False,
    progress=False
)

adj_close = prices['Adj Close']

#compute returns
returns = adj_close.pct_change().dropna()

if RETURN_FREQ == 'monthly':
    returns = (1 + returns).resample('ME').prod() - 1

#seperate market returns
market_returns = returns[MARKET_PROXY]

#estimate capm betas
betas = {}

for stock in MAG7:
    y = returns[stock]
    x = sm.add_constant(market_returns)
    model = sm.OLS(y, x).fit()
    betas[stock] = model.params[MARKET_PROXY]

#results
beta_df = pd.DataFrame.from_dict(
    betas,
    orient='index',
    columns=['CAPM Beta']
).sort_values('CAPM Beta', ascending=False)


#Expected betas for Mag 7 stocks
expected_beta = [
    {"Ticker": "TSLA", "Beta": 2.25},
    {"Ticker": "NVDA", "Beta": 1.86},
    {"Ticker": "META", "Beta": 1.38},
    {"Ticker": "AMZN", "Beta": 1.32},
    {"Ticker": "AAPL", "Beta": 1.25},
    {"Ticker": "GOOGL", "Beta": 1.03},
    {"Ticker": "MSFT", "Beta": 0.88}
]
expected_df = pd.DataFrame(expected_beta)
expected_df = expected_df.set_index('Ticker').rename(columns={'Beta': 'CAPM Beta'}).rename_axis(None)
expected_df = expected_df.sort_values('CAPM Beta', ascending=False)

print("Calculated beta:")
print(beta_df)
print("\nyfinance given beta:")
print(expected_df)

Calculated beta:
       CAPM Beta
TSLA    1.809278
NVDA    1.767862
AMZN    1.243296
AAPL    1.215888
META    1.151086
GOOGL   0.996681
MSFT    0.991967

yfinance given beta:
       CAPM Beta
TSLA        2.25
NVDA        1.86
META        1.38
AMZN        1.32
AAPL        1.25
GOOGL       1.03
MSFT        0.88


#Conclusions from previous test

*Hypothesis Testing*

Let $\beta_i$ be your estimated CAPM beta and $\beta_i^{(ref)}$ be Yahoo Finance’s reported beta.$$\begin{aligned}
H_0 &: \beta_i = \beta_i^{(ref)} \\
H_1 &: \beta_i \neq \beta_i^{(ref)}
\end{aligned}$$Decision Rule If $\beta_i^{(ref)}$ lies inside your $95\%$ confidence interval $\rightarrow$ Fail to reject $H_0$If $\beta_i^{(ref)}$ lies outside $\rightarrow$ Reject $H_0$

In [44]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats

MAG7 = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'TSLA']
MARKET_PROXY = 'SPY'   # S&P 500
START_DATE = '2015-01-01'
END_DATE = '2025-01-01'
ALPHA = 0.05

#yfinance reported betas for reference
YAHOO_BETAS = {
    'AAPL': 1.25,
    'MSFT': 0.88,
    'GOOGL': 1.03,
    'AMZN': 1.32,
    'META': 1.38,
    'NVDA': 1.86,
    'TSLA': 2.25
}

#price data
tickers = MAG7 + [MARKET_PROXY]
prices = yf.download(tickers, start=START_DATE, end=END_DATE, auto_adjust=True)

#yfinance column structure
if isinstance(prices.columns, pd.MultiIndex):
    prices = prices['Close']

#compute returns
returns = prices.pct_change().dropna()

market_returns = returns[MARKET_PROXY]

#estimate betas + tests
results = []

for stock in MAG7:
    y = returns[stock]
    X = sm.add_constant(market_returns)
    
    model = sm.OLS(y, X).fit()
    
    beta_hat = model.params[MARKET_PROXY]
    beta_se = model.bse[MARKET_PROXY]
    
    #95% confidence interval
    ci_low, ci_high = model.conf_int(alpha=ALPHA).loc[MARKET_PROXY]
    
    #hypothesis test vs Yahoo beta
    beta_ref = YAHOO_BETAS[stock]
    t_stat = (beta_hat - beta_ref) / beta_se
    p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=model.df_resid))
    
    decision = "Fail to Reject H0" if p_value > ALPHA else "Reject H0"
    
    results.append([
        stock,
        beta_hat,
        beta_ref,
        ci_low,
        ci_high,
        p_value,
        decision
    ])

#results
results_df = pd.DataFrame(
    results,
    columns=[
        'Stock',
        'Estimated Beta',
        'Yahoo Beta',
        'CI Lower (95%)',
        'CI Upper (95%)',
        'p-value',
        'Decision'
    ]
)

results_df.set_index('Stock', inplace=True)
print(results_df.round(4))

[*********************100%***********************]  8 of 8 completed

       Estimated Beta  Yahoo Beta  CI Lower (95%)  CI Upper (95%)  p-value  \
Stock                                                                        
AAPL           1.1925        1.25          1.1510          1.2340   0.0067   
MSFT           1.2150        0.88          1.1792          1.2509   0.0000   
GOOGL          1.1479        1.03          1.1045          1.1914   0.0000   
AMZN           1.1491        1.32          1.0929          1.2052   0.0000   
META           1.2739        1.38          1.2085          1.3394   0.0015   
NVDA           1.7353        1.86          1.6530          1.8176   0.0030   
TSLA           1.4913        2.25          1.3803          1.6024   0.0000   

        Decision  
Stock             
AAPL   Reject H0  
MSFT   Reject H0  
GOOGL  Reject H0  
AMZN   Reject H0  
META   Reject H0  
NVDA   Reject H0  
TSLA   Reject H0  


For all Mag 7 stocks, the null hypothesis is rejected at the 5% significance level.

This indicates that our estimated betas statistically differ from the Yahoo Finance-reported betas.

The confidence intervals provide further evidence: the reported Yahoo Beta for each stock falls outside the 95% CI of our estimated beta.

Overall, our results provide a more precise, data-driven estimation of systematic risk for these stocks.